# NuFrost Local Evaluation Notebook

This notebook runs the local accuracy assessment for NuFrost, Zhu2015, and HANTS without Google Colab or Google Drive.

In [ ]:
from pathlib import Path
import os


PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CACHE_DIR = PROJECT_DIR / "data" / "local_cache"
DATA_DIR = PROJECT_DIR / "data" / "input"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"[Info] Project directory: {PROJECT_DIR}")
print(f"[Info] Cache directory: {CACHE_DIR}")
print(f"[Info] Data directory: {DATA_DIR}")

print(f"[Info] Changing working directory to: {PROJECT_DIR}")
os.chdir(str(PROJECT_DIR))

In [ ]:
import src.data_loader
import importlib
import pandas as pd
from IPython.display import display

import src.evaluation
from config import build_args

importlib.reload(src.evaluation)

importlib.reload(src.data_loader)


In [ ]:
# Modify these parameters manually
TARGET_LON = 91.2734
TARGET_LAT = 29.7904
TARGET_BAND = "BLUE"
HLS_DATA_DIR = PROJECT_DIR / "data/hls"

IMAGE_NAMES = []

if not IMAGE_NAMES:
    from src.data_loader import find_image_chunks
    image_paths = find_image_chunks(str(HLS_DATA_DIR), TARGET_LON, TARGET_LAT, TARGET_BAND)
    image_paths_list = [image_paths] if image_paths else []
    print(f"[Info] Auto-detected {len(image_paths)} VRT chunk(s).")
else:
    image_paths_list = [[str(DATA_DIR / name)] for name in IMAGE_NAMES]
    print(f"[Info] Number of images to evaluate: {len(IMAGE_NAMES)}")


In [ ]:
print("========== Starting Local Accuracy Assessment ==========")

all_results = []
for image_paths in image_paths_list:
    first_path = Path(image_paths[0])
    image_name = first_path.name

    print(f"\n--- Evaluating: {image_name} ---")

    args = build_args({})
    args.image = image_paths
    args.cache_dir = str(CACHE_DIR)
    args.n_jobs = -1

    df_results = src.evaluation.evaluate_algorithms(
        image_path=args.image,
        args=args,
        num_points=40000,
        n_jobs=args.n_jobs
    )

    df_results.insert(0, "Image", image_name)
    all_results.append(df_results)

if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 1200)
    display(final_df)
else:
    print("No valid images processed.")
